# THETA — Precipitation Forecast Pipeline

End-to-end notebook for multi-model precipitation forecasting on NASA POWER data.

Run from the project root with:
```bash
jupyter notebook notebooks/Precipitation_Forecast_Pipeline.ipynb
```

Or execute programmatically:
```bash
python run_forecast.py
```

## 1. Imports

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath('.'))))

from utils.precipitation_models import (
    load_precipitation_data, load_from_json,
    build_features, split_train_test,
    sarima_fit, sarima_test_predictions, sarima_forward_forecast,
    train_xgboost, forecast_xgboost, evaluate,
    save_pipeline, save_forecast_to_json,
)
from utils.plotting import plot_forecast

## 2. Load Data

Fetch or load from existing Excel/JSON files.

In [ ]:
df, source_file = load_precipitation_data("jakarta", years=10)
print(f"Loaded {len(df)} days from {source_file}")
df.head()

## 3. Feature Engineering & Train/Test Split

In [ ]:
feat_df = build_features(df)
train_df, test_df = split_train_test(feat_df)
print(f"Train: {len(train_df)} | Test: {len(test_df)}")

## 4. Fit Models & Evaluate

In [ ]:
sarima_res = sarima_fit(train_df)
sarima_pred, sarima_ci = sarima_test_predictions(sarima_res, test_df)
sarima_m = evaluate(test_df['precip'].values, sarima_pred, 'SARIMAX')
print(f'SARIMAX: {sarima_m}')

xgb_model, feat_cols = train_xgboost(train_df)
xgb_pred = forecast_xgboost(xgb_model, feat_cols, test_df)
xgb_m = evaluate(test_df['precip'].values, xgb_pred, 'XGBoost')
print(f'XGBoost: {xgb_m}')

## 5. 7-Day Forecast

In [ ]:
from datetime import timedelta
import pandas as pd, numpy as np

fc_dates = pd.date_range(start=df.index[-1] + timedelta(days=1), periods=7, freq='D')
sarima_fc, sarima_ci = sarima_forward_forecast(sarima_res, feat_df, horizon=7)
xgb_fc = np.maximum(xgb_model.predict(feat_df[feat_cols].tail(7).values), 0)

fc = pd.DataFrame({'Date': fc_dates, 'SARIMAX': np.round(sarima_fc, 2), 'XGBoost': np.round(xgb_fc, 2)})
fc[['SARIMAX_Lo95', 'SARIMAX_Hi95']] = np.round(sarima_ci, 2)
fc

## 6. Plot & Save

In [ ]:
plot_forecast(
    train_df=train_df, test_df=test_df,
    sarima_pred=sarima_pred, sarima_test_ci=sarima_ci,
    xgb_pred=xgb_pred,
    fc=fc, city='Jakarta'
)
save_pipeline(sarima_res, xgb_model, feat_cols, {'city': 'jakarta'}, 'data/processed/precipitation_model.joblib')
save_forecast_to_json('Jakarta', fc_dates, sarima_fc, xgb_fc, sarima_ci, [sarima_m, xgb_m], 'outputs/predictions/forecast_Jakarta.json')